# Challenge 1: datos y análisis exploratorio

Tu primera responsabilidad es preparar una fuente de datos confiable para el proyecto de clasificación de vinos.

**Entrada:** `data/raw/wine.csv`  
**Salidas:** `data/processed/train.csv`, `data/processed/test.csv` y `artifacts/data_contract.json`

> El conjunto de test se crea aquí, pero no debe explorarse ni utilizarse después del split.

## Objetivos

- cargar y validar datos externos;
- identificar problemas básicos de calidad;
- crear un split reproducible y estratificado;
- realizar EDA únicamente sobre entrenamiento;
- guardar datos y metadatos para la siguiente etapa.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
from pathlib import Path
PROJECT_DIR = Path.cwd()
if (PROJECT_DIR / "Lectura 6").is_dir():
    PROJECT_DIR = PROJECT_DIR / "Lectura 6"
RAW_DATA_DIR = PROJECT_DIR / "Data" if (PROJECT_DIR / "Data").is_dir() else PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


## 1. Carga y profiling

In [ ]:
DATA_PATH = RAW_DATA_DIR / "wine.csv"
df = pd.read_csv(DATA_PATH)
print("Datos cargados:", DATA_PATH)
print(df.head())


In [ ]:
print("Forma:", df.shape)
print("Tipos:\n", df.dtypes)
print("Nulos por columna:\n", df.isna().sum())
print("Duplicados:", int(df.duplicated().sum()))
print("Distribución de target:\n", df["target"].value_counts().sort_index())


**Conclusión de calidad:**

> Describe brevemente qué encontraste y qué decisión tomaste.

## 2. Split train/test

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["target"],
    random_state=RANDOM_STATE,
)


In [ ]:
# Verificación del split
assert len(train_df) + len(test_df) == len(df)
assert set(train_df.index).isdisjoint(set(test_df.index))
print("Train:", train_df.shape)
print("Test:", test_df.shape)

## 3. EDA sobre entrenamiento

Incluye:

- distribución de clases;
- distribuciones o boxplots de variables relevantes;
- correlaciones;
- tres hallazgos que puedan influir en el modelado.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
train_df["target"].value_counts().sort_index().plot.bar(ax=axes[0], title="Distribución de clases")
train_df.groupby("target")["alcohol"].plot(legend=True, ax=axes[1])
axes[1].set_title("Alcohol por clase")
train_df.boxplot(column="proline", by="target", ax=axes[2])
axes[2].set_title("Proline por clase")
axes[2].set_xlabel("target")
plt.suptitle("")
plt.tight_layout()
plt.show()


**Hallazgos:**

1.
2.
3.

## 4. Persistencia de datos y contrato

In [ ]:
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train_df.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)
contract = {
    "target": "target",
    "features": [column for column in df.columns if column != "target"],
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
}
(ARTIFACTS_DIR / "data_contract.json").write_text(
    json.dumps(contract, indent=2), encoding="utf-8"
)
print("Salidas guardadas en", PROCESSED_DATA_DIR)


In [ ]:
# Verificación de entrega
assert (PROCESSED_DATA_DIR / "train.csv").exists()
assert (PROCESSED_DATA_DIR / "test.csv").exists()
assert (ARTIFACTS_DIR / "data_contract.json").exists()
print("Challenge 1 completado.")

## Checklist

- [ ] Cargué el CSV desde `data/raw`.
- [ ] Revisé calidad y distribución de la clase.
- [ ] Realicé un split estratificado.
- [ ] No exploré el conjunto de test.
- [ ] Guardé los tres artefactos requeridos.